# **La creation du fichier ventes dirty CSV**


In [ ]:
import random
import csv

random.seed(42)

header = ["ID", "Prix", "Quantite", "Remise"]
rows = []

for i in range(100):
    row_id = 100 + i
    
    # Prix : manquant, virgule française, avec €, négatif, ou normal
    r = random.random()
    if r < 0.05:
        prix = ""
    elif r < 0.10:
        prix = str(round(random.uniform(5, 200), 2)).replace(".", ",")
    elif r < 0.13:
        prix = f"{round(random.uniform(5, 200), 2)} €"
    elif r < 0.15:
        prix = f"-{round(random.uniform(5, 50), 2)}"
    else:
        prix = round(random.uniform(5, 200), 2)
    
    # Quantite : manquant, zéro, en lettres, outlier, ou normal
    r = random.random()
    if r < 0.04:
        quantite = ""
    elif r < 0.07:
        quantite = 0
    elif r < 0.10:
        quantite = random.choice(["un", "deux", "trois", "cinq"])
    elif r < 0.12:
        quantite = random.randint(100, 500)
    else:
        quantite = random.randint(1, 15)
    
    # Remise : manquante, avec %, hors plage, négative, ou normale
    r = random.random()
    if r < 0.08:
        remise = ""
    elif r < 0.12:
        remise = f"{random.randint(0, 50)}%"
    elif r < 0.14:
        remise = random.randint(101, 150)
    elif r < 0.16:
        remise = -random.randint(1, 10)
    else:
        remise = random.choice([0, 0, 0, 5, 5, 10, 10, 15, 20, 25, 30])
    
    rows.append([row_id, prix, quantite, remise])

# Injecter des IDs dupliqués
rows[30][0] = rows[10][0]
rows[75][0] = rows[22][0]

with open("ventes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rows)

print("Fichier ventes.csv créé avec 100 lignes de données sales.")

Fichier ventes.csv créé avec 100 lignes de données sales.


# **Nettoyage du dataset**

**Chargement & inspection**


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('ventes.csv')
print(df.shape)
df.head(10)

(100, 4)


,ID,Prix,Quantite,Remise
0,100,9.88,4,107
1,101,178.98,cinq,NaN
2,102,"50,37",9,30
3,103,86.81,5,0
4,104,36.13,5,-6
5,105,79.09 €,6,0
6,106,109.56,7,NaN
7,107,127.58,6,0
8,108,NaN,5,5
9,109,79.12,14,10


**Info & valeurs manquantes**

In [2]:
df.info()
print('\nNaN par colonne:')
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   ID        100 non-null    int64
 1   Prix      98 non-null     str  
 2   Quantite  99 non-null     str  
 3   Remise    90 non-null     str  
dtypes: int64(1), str(3)
memory usage: 3.3 KB

NaN par colonne:
ID           0
Prix         2
Quantite     1
Remise      10
dtype: int64


**Reperer les valeurs suspects**

In [3]:
for col in ['Prix', 'Quantite', 'Remise']:
    print(f'--- {col} ---')
    print(df[col].astype(str).unique()[:15])
    print()

--- Prix ---
<StringArray>
[   '9.88',  '178.98',   '50,37',   '86.81',   '36.13', '79.09 €',  '109.56',
  '127.58',       nan,   '79.12',  '135.68',  '109.16',   '47.82',  '161.98',
   '66.36']
Length: 15, dtype: str

--- Quantite ---
<StringArray>
[   '4', 'cinq',    '9',    '5',    '6',    '7',   '14',   '11',    '8',
   '13',    '2',    '3',   '10',    '0',    nan]
Length: 15, dtype: str

--- Remise ---
<StringArray>
['107', nan, '30', '0', '-6', '5', '10', '25', '20', '42%', '15', '27%', '1%']
Length: 13, dtype: str



**Nettoyer Prix**

In [4]:
df_clean = df.copy()

df_clean['Prix'] = (
    df_clean['Prix'].astype(str)
    .str.replace('€', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df_clean['Prix'] = pd.to_numeric(df_clean['Prix'], errors='coerce')
df_clean.loc[df_clean['Prix'] < 0, 'Prix'] = df_clean['Prix'].abs()

**Nettoyer Quantite**

In [5]:
mots_vers_int = {'un': 1, 'deux': 2, 'trois': 3, 'quatre': 4, 'cinq': 5}

df_clean['Quantite'] = (
    df_clean['Quantite'].astype(str).str.strip().str.lower()
    .replace(mots_vers_int)
)
df_clean['Quantite'] = pd.to_numeric(df_clean['Quantite'], errors='coerce')

**Nettoyer Remise**

In [6]:
df_clean['Remise'] = (
    df_clean['Remise'].astype(str)
    .str.replace('%', '', regex=False)
    .str.strip()
)
df_clean['Remise'] = pd.to_numeric(df_clean['Remise'], errors='coerce')
df_clean.loc[(df_clean['Remise'] < 0) | (df_clean['Remise'] > 100), 'Remise'] = np.nan

**Doublons**

In [7]:
print('IDs dupliqués :')
print(df_clean[df_clean.duplicated('ID', keep=False)].sort_values('ID'))

df_clean = df_clean.drop_duplicates('ID', keep='first').reset_index(drop=True)

IDs dupliqués :
     ID    Prix  Quantite  Remise
10  110  135.68      11.0    25.0
30  110  119.75       NaN     0.0
22  122  103.99       6.0    20.0
75  122  181.05       5.0     0.0


**Imputation**

In [8]:
df_clean['Prix'] = df_clean['Prix'].fillna(df_clean['Prix'].median())
df_clean['Quantite'] = df_clean['Quantite'].fillna(df_clean['Quantite'].median()).astype(int)
df_clean['Remise'] = df_clean['Remise'].fillna(0)

df_clean.info()
df_clean.describe()

<class 'pandas.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ID        98 non-null     int64  
 1   Prix      98 non-null     float64
 2   Quantite  98 non-null     int64  
 3   Remise    98 non-null     float64
dtypes: float64(2), int64(2)
memory usage: 3.2 KB


,ID,Prix,Quantite,Remise
count,98.000000,98.000000,98.000000,98.000000
mean,149.438776,116.401020,8.938776,9.897959
std,29.127189,56.685785,12.096533,10.766012
min,100.000000,7.240000,0.000000,0.000000
25%,124.250000,69.067500,4.250000,0.000000
50%,149.500000,119.950000,7.500000,5.000000
75%,173.750000,172.545000,12.000000,20.000000
max,199.000000,199.860000,119.000000,42.000000


**Export simple**

In [9]:
df_clean.to_csv('ventes_clean.csv', index=False)
print('Export OK :', df_clean.shape)
df_clean.head()


Export OK : (98, 4)


,ID,Prix,Quantite,Remise
0,100,9.88,4,0.0
1,101,178.98,5,0.0
2,102,50.37,9,30.0
3,103,86.81,5,0.0
4,104,36.13,5,0.0
